In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\adaly\OneDrive\Documents\DMDeficientGalaxyTNG-1")
CATALOG = PROJECT_DIR / "data" / "dmdg_catalog_z0.csv"
df = pd.read_csv(CATALOG)

print(f"Total catalog objects: {len(df):,}")
resolved = df[
    (df["N_star"] >= 500) &
    (df["SubhaloFlag"] == 1) &
    (df["f_DM"].notna())
].copy()

print(f"Resolved galaxies: {len(resolved):,}")
centrals = resolved[
    resolved["IsSatellite"] == False
].copy()

satellites = resolved[
    (resolved["IsSatellite"] == True) &
    resolved[["x", "y", "z"]].notna().all(axis=1)
].copy()

nearest_distances = []

for _, central in centrals.iterrows():

    sats = satellites[
        satellites["GroupID"] == central["GroupID"]
    ]

    if len(sats) == 0:
        nearest_distances.append(np.nan)
        continue

    box_size = 205000.0  # TNG300-1 box size in ckpc/h

    dx = np.abs(sats["x"].to_numpy() - central["x"])
    dy = np.abs(sats["y"].to_numpy() - central["y"])
    dz = np.abs(sats["z"].to_numpy() - central["z"])

    dx = np.minimum(dx, box_size - dx)
    dy = np.minimum(dy, box_size - dy)
    dz = np.minimum(dz, box_size - dz)

    distances = np.sqrt(dx**2 + dy**2 + dz**2)


    nearest_distances.append(np.min(distances))

centrals["NearestSatelliteDistance"] = nearest_distances
dmdgs = centrals[
    centrals["f_DM"] < 0.5
].copy()
dmdgs_non_extreme = dmdgs[
   dmdgs["f_DM"] > 0.05
].copy()
print(f"Centrals: {len(centrals):,}")
print(f"DMDGS (total): {len(dmdgs):,}")
print(f"DMDGS (non-tidal stripping): {len(dmdgs_non_extreme):,}")
print(f"DMDGS Fraction: {(len(dmdgs_non_extreme) / len(centrals)):,}")


Total catalog objects: 13,922,457
Resolved galaxies: 154,638
Centrals: 93,039
DMDGS (total): 96
DMDGS (non-tidal stripping): 96
DMDGS Fraction: 0.0010318253635572179


In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint
from sklearn.tree import export_graphviz
from IPython.display import Image
import graphviz

In [6]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
controls = centrals[
    centrals["f_DM"] >= 0.5
].copy()
print(centrals.keys())
dmdgs = dmdgs_non_extreme.copy()
controls["logMstar"] = np.log10(controls["M_star"])
controls["logMhost"] = np.log10(controls["Host_M200c"])

dmdgs["logMstar"] = np.log10(dmdgs["M_star"])
dmdgs["logMhost"] = np.log10(dmdgs["Host_M200c"])
match_features = ["logMstar", "logMhost"]
scaler = StandardScaler()

X_control = scaler.fit_transform(
    controls[match_features]
)

X_dmdg = scaler.transform(
    dmdgs[match_features]
)
nn = NearestNeighbors(n_neighbors=3)
nn.fit(X_control)

distances, indices = nn.kneighbors(X_dmdg)

print("Matching successful!")
print("DMDGs:", len(dmdgs))
print("Controls:", len(controls))
print("Distance array shape:", distances.shape)
print("Index array shape:", indices.shape)

Index(['SubhaloID', 'GroupID', 'M_DM_2Rh', 'M_total_2Rh', 'f_DM', 'M_star',
       'M_gas', 'N_gas', 'N_DM', 'N_star', 'N_BH', 'R_half_star', 'x', 'y',
       'z', 'SFR', 'Vmax', 'VmaxRad', 'SubhaloFlag', 'Host_M200c',
       'Host_R200c', 'IsCentral', 'IsSatellite', 'DistanceToHost',
       'r_over_R200c', 'DMDG_fDM_lt_0.5', 'NearestSatelliteDistance'],
      dtype='str')
Matching successful!
DMDGs: 96
Controls: 92943
Distance array shape: (96, 3)
Index array shape: (96, 3)


In [7]:
print(dmdgs)
print(controls)
fullDataSet = pd.concat([dmdgs, controls], ignore_index=True)
print(fullDataSet)

         SubhaloID  GroupID   M_DM_2Rh  M_total_2Rh      f_DM     M_star  \
809044      809044     1473  19.662199    39.464733  0.498222  31.312744   
852697      852697     1701  10.830939    23.440784  0.462055  18.955132   
918341      918341     2102   5.469246    11.354618  0.481676   8.159344   
965175      965175     2442  11.113763    26.537890  0.418789  23.042484   
1025250    1025250     2944   5.708252    12.088194  0.472217   9.420410   
...            ...      ...        ...          ...       ...        ...   
1980857    1980857    53706   1.115360     2.436473  0.457776   1.593023   
1990303    1990303    55154   0.450127     0.918615  0.490006   0.549314   
2012519    2012519    58729   0.991873     1.989062  0.498664   1.208134   
2049968    2049968    65161   0.374442     1.019240  0.367374   0.838199   
2259153    2259153   113055   0.310707     0.783788  0.396417   0.542708   

              M_gas   N_gas    N_DM  N_star  ...   Host_M200c  Host_R200c  \
809044   2

In [8]:
from sklearn.model_selection import train_test_split
fullDataSet["gas_fraction"] = (
    fullDataSet["M_gas"] /
    (fullDataSet["M_star"] + fullDataSet["M_gas"])
)
features = [
    "logMstar",
    "logMhost",
    "NearestSatelliteDistance",
    "gas_fraction",
    "SFR",
    "Vmax"
]

X = fullDataSet[features]
Y = fullDataSet["f_DM"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)


In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Random Forest results:")
print(f"R²   = {r2:.4f}")
print(f"MAE  = {mae:.4f}")
print(f"RMSE = {rmse:.4f}")


Random Forest results:
R²   = 0.4970
MAE  = 0.0394
RMSE = 0.0500


In [11]:
predictions = X_test.copy()
predictions["actual_f_DM"] = y_test
predictions["predicted_f_DM"] = y_pred
predictions["residual"] = (
    predictions["actual_f_DM"] -
    predictions["predicted_f_DM"]
)
predictionsDMDG = predictions[
   predictions["actual_f_DM"] < 0.5
].copy()
averageNonAbsolute = predictionsDMDG["residual"].median()
print(averageNonAbsolute)
predictionsNonDMDG = predictions[
   predictions["actual_f_DM"] > 0.5
]
averageNonAbsolute = predictionsNonDMDG["residual"].median()
print(averageNonAbsolute)

-0.12246187084000065
0.0010969868399994809


In [12]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

print(feature_importance)

                    Feature  Importance
0                  logMstar    0.257761
4                       SFR    0.251318
1                  logMhost    0.237488
5                      Vmax    0.130301
3              gas_fraction    0.096288
2  NearestSatelliteDistance    0.026843
